In [1]:
!nvidia-smi

Thu Jun 12 09:56:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             39W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%capture
%pip install lightning
%pip install adversarial-robustness-toolbox[pytorch_image]

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import torchvision.models as models
import torchmetrics
import lightning as L
from lightning import LightningModule, LightningDataModule
import numpy as np

# ART imports
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

In [4]:
# Configuration
EPOCHS = 10

BATCH_SIZE = 128
LEARNING_RATE = 1e-3
SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Set seed
torch.manual_seed(SEED)
np.random.seed(SEED)

In [5]:
class AdversarialResNet18(LightningModule):
    def __init__(self, num_classes=2, attack_type="fgsm", epsilon=0.01):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        
        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy(task="binary", num_classes=num_classes)
        
        self.attack_type = attack_type
        self.epsilon = epsilon
        self.save_hyperparameters()
        
        # Initialize ART classifier (will be set up in training_step)
        self.art_classifier = None
        
    def forward(self, x):
        return self.model(x)
    
    def setup_art_classifier(self):
        """Setup ART classifier for adversarial attacks"""
        if self.art_classifier is None:
            self.art_classifier = PyTorchClassifier(
                model=self.model,
                loss=self.criterion,
                input_shape=(3, 224, 224),  # Adjust based on your input size
                nb_classes=2,
                clip_values=(0.0, 1.0)
            )
            
            # Setup attack
            if self.attack_type == "fgsm":
                self.attack = FastGradientMethod(
                    estimator=self.art_classifier,
                    eps=self.epsilon
                )
            elif self.attack_type == "pgd":
                self.attack = ProjectedGradientDescent(
                    estimator=self.art_classifier,
                    eps=self.epsilon,
                    eps_step=0.01,
                    max_iter=10
                )
    
    def generate_adversarial_examples(self, x, y):
        """Generate adversarial examples using ART"""
        self.setup_art_classifier()
        
        # Convert to numpy for ART
        x_np = x.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        
        # Generate adversarial examples
        x_adv = self.attack.generate(x=x_np, y=y_np)
        
        # Convert back to tensor
        x_adv_tensor = torch.from_numpy(x_adv).to(self.device)
        
        return x_adv_tensor
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # Standard training loss
        logits_clean = self(x)
        loss_clean = self.criterion(logits_clean, y)
        
        # Generate adversarial examples
        x_adv = self.generate_adversarial_examples(x, y)
        
        # Adversarial training loss
        logits_adv = self(x_adv)
        loss_adv = self.criterion(logits_adv, y)
        
        # Combined loss (clean + adversarial)
        total_loss = 0.5 * loss_clean + 0.5 * loss_adv
        
        # Accuracy on clean examples
        preds_clean = torch.argmax(logits_clean, dim=1)
        self.train_acc(preds_clean, y)
        
        # Logging
        self.log('train_loss', total_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_loss_clean', loss_clean, on_step=False, on_epoch=True)
        self.log('train_loss_adv', loss_adv, on_step=False, on_epoch=True)
        self.log('train_acc', self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        
        preds = torch.argmax(logits, dim=1)
        self.val_acc(preds, y)
        
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_acc', self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

In [6]:
class PCAMDataset(torch.utils.data.Dataset):
    def __init__(self, input_file_path, label_file_path, transform=None, lazy=False, max_samples=None):
        import h5py
        from PIL import Image
        from tqdm.notebook import tqdm
        
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform
        self.lazy = lazy
        self.max_samples = max_samples
        
        input_files = h5py.File(self.input_file_path)["x"]
        self.input_files = input_files
        target_files = h5py.File(self.label_file_path)["y"]
        self.target_files = target_files
        
        if max_samples is not None:
            self.length = min(max_samples, len(input_files))
        else:
            self.length = len(input_files)
        
        if not lazy:
            self.images = [Image.fromarray(input_files[i]).convert("RGB") for i in tqdm(range(self.length))]
            self.targets = [int(target_files[i, 0, 0, 0]) for i in tqdm(range(self.length))]
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError("Index out of range")
        
        if not self.lazy:
            image, target = self.images[idx], torch.tensor(self.targets[idx])
        else:
            from PIL import Image
            image = Image.fromarray(self.input_files[idx]).convert("RGB")
            target = int(self.target_files[idx, 0, 0, 0])
        
        if self.transform:
            image = self.transform(image)
        return image, target

In [7]:
# Simple transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Create datasets (adjust paths as needed)
train_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5",
    transform=train_transform,
    max_samples=5000  # Reduced for faster training
)

val_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5",
    transform=eval_transform,
    max_samples=1000
)

test_data = PCAMDataset(
    input_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5",
    label_file_path="/kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5",
    transform=eval_transform,
    max_samples=1000
)

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
# DataModule
class SimpleDataModule(LightningDataModule):
    def __init__(self, train_dataset, val_dataset, batch_size=BATCH_SIZE):
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.batch_size = batch_size
    
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=2)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=2)

In [9]:
# Training function
def train_adversarial_model(attack_type="fgsm", epsilon=0.1, checkpoint_path=None):
    print(f"Training model with {attack_type.upper()} adversarial training (eps={epsilon})")

    # Create model
    if checkpoint_path is not None:
        print(f"Loading model from checkpoint: {checkpoint_path}")
        model = AdversarialResNet18.load_from_checkpoint(
            checkpoint_path,
            num_classes=2,
            attack_type=attack_type,
            epsilon=epsilon
        )
    else:
        model = AdversarialResNet18(
            num_classes=2, 
            attack_type=attack_type, 
            epsilon=epsilon
        )



    # Create data module
    datamodule = SimpleDataModule(train_data, val_data)
    
    # Create trainer
    trainer = L.Trainer(
        max_epochs=EPOCHS,
        accelerator='auto',
        devices=1,
        precision=16,  # Mixed precision for faster training
        log_every_n_steps=50
    )
    
    # Train
    trainer.fit(model, datamodule)
    
    return model

In [10]:
# Train FGSM model
checkpoint_path='/kaggle/input/hermes_resnet_18/pytorch/no_augmentation_lightning/1/reduced_dataset_no_data_augmentation.ckpt'

fgsm_model = train_adversarial_model(attack_type="fgsm", epsilon=0.1,checkpoint_path=checkpoint_path)

# Train PGD model
#gd_model = train_adversarial_model(attack_type="pgd", epsilon=0.1)

Training model with FGSM adversarial training (eps=0.1)
Loading model from checkpoint: /kaggle/input/hermes_resnet_18/pytorch/no_augmentation_lightning/1/reduced_dataset_no_data_augmentation.ckpt


/usr/local/lib/python3.11/dist-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO: Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
2025-06-12 09:56:26.414774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749722186.433488     379 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749722186.438864     379 cuda_blas.cc:1418] Unable 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (40) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.


In [16]:
import shutil

run_name = "resnet18_adversarial_training_fgsm_epsilon_0.1"

# Salva i pesi in un file .pth
pth_path = "/kaggle/working/" + run_name + ".pth"
torch.save(fgsm_model.model.state_dict(), pth_path)

# Copia lo stesso file .pth in un percorso .ckpt
ckpt_path = "/kaggle/working/" + run_name + ".ckpt"
shutil.copy2(pth_path, ckpt_path)

'/kaggle/working/resnet18_adversarial_training_fgsm_epsilon_0.1.ckpt'